<div style='width:100%; display:flex; justify-content:space-between; align-items:center; margin: 1em 0;'>
  <a href='3.%20first_steps_with_psql.ipynb' style='padding:6px 14px; background:#007bff; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>← Previous</a>
  <a href='../TOC.md' style='padding:6px 14px; background:#6c757d; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>📚 Table of Contents</a>
  <a href='../2.%20SQL_essentials/5.%20data_types_done_right.ipynb' style='padding:6px 14px; background:#28a745; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>Next →</a>
</div>

# Chapter 4: Core Concepts You Must Get Right Early

Some PostgreSQL concepts, if misunderstood in the first week, cause problems that multiply for years. This chapter addresses the cluster/database/schema hierarchy, system catalogs, physical storage, and naming conventions — the invisible scaffolding that every PostgreSQL operation depends on.

## The Big Picture

Imagine a city's postal system. A **cluster** is the country — one postal system serving everyone. A **database** is a state — mail cannot cross state lines without special forwarding (foreign data wrappers). A **schema** is a city — organising addresses into neighbourhoods. A **table** is a building — where the data actually lives.

This hierarchy has profound consequences. Connection strings identify a cluster and database. You cannot JOIN tables from two different databases without an extension. Granting privileges on a database does not grant access to its tables — each schema and object has independent access control. These constraints are not bugs; they are security boundaries.

The `pg_catalog` and `information_schema` views are PostgreSQL's nervous system — they describe everything about the database itself. Knowing how to query them means you can answer 'what indexes exist?', 'who owns this table?', 'what constraints are on this column?' without a GUI.

Physical storage concepts — pages, TOAST, the visibility map — are not just academic. They explain why large TEXT columns do not slow down queries that do not select them (TOAST), why index-only scans are sometimes possible (visibility map), and why bulk updates need batching (dead tuple accumulation).

## Core Concepts

### The Cluster / Database / Schema / Object Hierarchy
One PostgreSQL process = one cluster = one port = one `PGDATA`. A cluster holds multiple databases. Databases hold schemas. Schemas hold tables, views, functions, sequences, and types. Objects cannot be queried across database boundaries without `postgres_fdw`.

### The `public` Schema Problem
Before PostgreSQL 15, all users had `CREATE` privileges on `public` by default. Industry standard: revoke those privileges immediately, create an application-specific schema, and set `search_path` explicitly.

### `pg_catalog` vs `information_schema`
`pg_catalog` is PostgreSQL's native metadata system — fast, comprehensive, PostgreSQL-specific. `information_schema` is the SQL-standard portable interface — slower but portable across vendors. Use `pg_catalog` for operational scripts; `information_schema` for portable tooling.

### TOAST (The Oversized-Attribute Storage Technique)
Columns with values larger than ~2KB are stored in a separate TOAST table with a pointer in the main row. TOAST strategies: PLAIN (no toast), EXTENDED (compress+toast, default for TEXT/JSONB), EXTERNAL (toast without compression), MAIN (compress in-place first).

### Naming Conventions
PostgreSQL folds unquoted identifiers to lowercase. Industry standard: `snake_case` for everything, no quoted identifiers, named constraints, singular table names, consistent column suffixes (`_id`, `_at`, `_count`).

## Key SQL Syntax

```sql
-- Schema security baseline
REVOKE CREATE ON SCHEMA public FROM PUBLIC;
CREATE SCHEMA IF NOT EXISTS app;
ALTER DATABASE appdb SET search_path TO app, public;

-- Cluster information
SELECT pg_postmaster_start_time(), inet_server_port(),
       current_setting('data_directory');

-- pg_catalog metadata query
SELECT c.relname, n.nspname, pg_size_pretty(pg_table_size(c.oid))
FROM pg_class c JOIN pg_namespace n ON n.oid = c.relnamespace
WHERE c.relkind = 'r' AND n.nspname = 'app'
ORDER BY pg_table_size(c.oid) DESC;

-- Check TOAST strategy per column
SELECT attname, CASE attstorage
    WHEN 'p' THEN 'PLAIN' WHEN 'e' THEN 'EXTERNAL'
    WHEN 'x' THEN 'EXTENDED' WHEN 'm' THEN 'MAIN'
END AS toast_strategy
FROM pg_attribute
WHERE attrelid = 'app.documents'::regclass AND attnum > 0;
```

In [ ]:
# The cluster/database/schema/object hierarchy

hierarchy = {
    'Cluster (Instance)': {
        'description': 'One postmaster process, one PGDATA, one port',
        'scope': 'Global — roles, tablespaces, replication slots live here',
        'cross_boundary': 'Cannot — you are always connected to ONE database',
        'example': 'PostgreSQL 16 on port 5432',
    },
    'Database': {
        'description': 'Named collection of schemas and objects',
        'scope': 'Isolated — no cross-database JOINs without postgres_fdw',
        'cross_boundary': 'Requires Foreign Data Wrapper for cross-DB queries',
        'example': 'appdb, appdb_staging, monitoring',
    },
    'Schema': {
        'description': 'Namespace within a database (like a folder)',
        'scope': 'Organises objects; separate access control per schema',
        'cross_boundary': 'Can JOIN across schemas freely within same database',
        'example': 'app, extensions, migrations, audit',
    },
    'Object': {
        'description': 'Table, view, function, sequence, type',
        'scope': 'Fully qualified: schema.object_name',
        'cross_boundary': 'Always qualify with schema in production code',
        'example': 'app.users, app.orders, extensions.crypt()',
    },
}

print('=== PostgreSQL Hierarchy ===')
for level, info in hierarchy.items():
    print(f'  [{level}]')
    for key, value in info.items():
        print(f'    {key:<16} {value}')
    print()

schema_setup = [
    '-- Security baseline: revoke default public create access',
    'REVOKE CREATE ON SCHEMA public FROM PUBLIC;',
    'REVOKE ALL ON SCHEMA public FROM PUBLIC;',
    '',
    '-- Create application schema',
    'CREATE SCHEMA IF NOT EXISTS app;',
    "COMMENT ON SCHEMA app IS 'Application tables — Platform Team';",
    '',
    '-- Extension schema (keeps pg_catalog cleaner)',
    'CREATE SCHEMA IF NOT EXISTS extensions;',
    'CREATE EXTENSION IF NOT EXISTS pgcrypto WITH SCHEMA extensions;',
    '',
    '-- Set default search path',
    "ALTER DATABASE appdb SET search_path TO app, extensions, public;",
]

print('=== Recommended Schema Setup ===')
for line in schema_setup:
    print(f'  {line}')

## Deep Dive with Examples

### Example 1: `pg_catalog` — The Native Metadata Interface

`pg_catalog` is PostgreSQL's internal system catalog. Every `\d` command in psql is a `pg_catalog` query. These queries are fast, comprehensive, and PostgreSQL-specific — use them for operational scripts.

In [ ]:
# Key pg_catalog queries

catalog_queries = {
    'Tables with sizes': (
        'SELECT n.nspname AS schema, c.relname AS table_name,\n'
        '       pg_size_pretty(pg_total_relation_size(c.oid)) AS total_size,\n'
        '       c.reltuples::BIGINT AS estimated_rows\n'
        'FROM pg_class c\n'
        'JOIN pg_namespace n ON n.oid = c.relnamespace\n'
        "WHERE c.relkind = 'r' AND n.nspname = 'app'\n"
        'ORDER BY pg_total_relation_size(c.oid) DESC;'
    ),
    'Unused indexes': (
        'SELECT s.schemaname, s.tablename, s.indexname,\n'
        '       pg_size_pretty(pg_relation_size(s.indexrelid)) AS size\n'
        'FROM pg_stat_user_indexes s\n'
        "WHERE s.idx_scan = 0 AND s.schemaname = 'app'\n"
        'ORDER BY pg_relation_size(s.indexrelid) DESC;'
    ),
    'Active connections and waits': (
        'SELECT pid, usename, state, wait_event_type, wait_event,\n'
        '       left(query, 80) AS query\n'
        'FROM pg_stat_activity\n'
        "WHERE state != 'idle' ORDER BY query_start;"
    ),
    'Foreign keys without indexes': (
        'SELECT tc.table_name, kcu.column_name\n'
        'FROM information_schema.table_constraints tc\n'
        'JOIN information_schema.key_column_usage kcu\n'
        '    ON tc.constraint_name = kcu.constraint_name\n'
        "WHERE tc.constraint_type = 'FOREIGN KEY'\n"
        '  AND NOT EXISTS (\n'
        '    SELECT 1 FROM pg_indexes pi\n'
        '    WHERE pi.tablename = tc.table_name\n'
        '      AND pi.indexdef LIKE \'%\' || kcu.column_name || \'%\'\n'
        '  );'
    ),
}

print('=== Key pg_catalog Queries ===')
for title, query in catalog_queries.items():
    print(f'  -- {title}')
    print(query)
    print()

print('=== When to Use Each Catalog ===')
catalog_choice = [
    ('pg_catalog',        'Operational scripts, performance tuning, admin tasks'),
    ('information_schema', 'ORM introspection, portable tooling, compliance reports'),
]
for catalog, use_case in catalog_choice:
    print(f'  {catalog:<22} {use_case}')

## Deep Dive with Examples

### Example 2: TOAST — Why Large Values Do Not Slow Down Your Queries

TOAST allows tables with large text or binary columns to remain fast for queries that do not access those large columns. The TOAST table is completely transparent to SQL — you never write TOAST-aware queries.

In [ ]:
# TOAST storage strategies explained

toast_strategies = {
    'PLAIN': {
        'description': 'Stored inline in the main table page — no TOAST',
        'when': 'Fixed-width types: INTEGER, BIGINT, BOOLEAN, FLOAT, DATE',
        'compression': 'Never',
        'external': 'Never',
        'change': 'Cannot change for fixed-width types',
    },
    'EXTENDED': {
        'description': 'Compress first; if still large, store in TOAST table',
        'when': 'TEXT, BYTEA, JSONB, ARRAY — variable-length types (default)',
        'compression': 'pglz or lz4',
        'external': 'Yes, when row exceeds ~2KB after compression',
        'change': 'ALTER TABLE t ALTER COLUMN c SET STORAGE EXTENDED;',
    },
    'EXTERNAL': {
        'description': 'Store in TOAST table without compression',
        'when': 'Pre-compressed data: JPEG, PNG, gzip stored as BYTEA',
        'compression': 'Never (data already compressed)',
        'external': 'Yes, when row exceeds ~2KB',
        'change': 'ALTER TABLE t ALTER COLUMN c SET STORAGE EXTERNAL;',
    },
    'MAIN': {
        'description': 'Compress inline first; external only as last resort',
        'when': 'Columns to keep local in main table if possible',
        'compression': 'pglz first',
        'external': 'Only when nothing else fits in the 8KB page',
        'change': 'ALTER TABLE t ALTER COLUMN c SET STORAGE MAIN;',
    },
}

print('=== TOAST Storage Strategies ===')
for strategy, info in toast_strategies.items():
    print(f'  {strategy}')
    for key, value in info.items():
        print(f'    {key:<12} {value}')
    print()

print('=== Key TOAST Insight ===')
print('  When you SELECT columns that are NOT the large TOAST column, PostgreSQL')
print('  reads only the main table pages and ignores the TOAST table entirely.')
print()
print('  A table with 1GB of JSONB documents can still have fast')
print('  SELECT id, email FROM users queries because JSONB is not accessed.')
print()
print('  Check TOAST table size:')
print('    SELECT relname, pg_size_pretty(pg_relation_size(reltoastrelid))')
print('    FROM pg_class WHERE relname = \'your_table\';')

## Deep Dive with Examples

### Example 3: Naming Conventions — The Invisible Architecture

Consistent naming is the lowest-cost, highest-leverage improvement you can make to a schema. It eliminates ambiguity during incidents, simplifies tooling, and documents intent without extra comments.

In [ ]:
# PostgreSQL naming conventions reference

naming_rules = [
    ('Case',       'snake_case always',
     'user_id, order_items, created_at',
     'UserId, OrderItems, "CamelCase"',
     'Quoted identifiers require quotes everywhere forever'),
    ('Tables',     'Singular nouns in snake_case',
     'user, order, product, order_item',
     'users, Orders, tbl_users',
     'Consistent; table represents entity type not collection'),
    ('Primary keys', 'table_name_id pattern',
     'user_id, order_id, product_id',
     'id, ID, pk, key',
     'Prevents collision in JOINs; self-documenting'),
    ('Timestamps', '_at suffix for events, _date for calendar dates',
     'created_at, updated_at, published_at, birth_date',
     'create_time, update_date, timestamp',
     'Distinguishes TIMESTAMPTZ events from DATE values'),
    ('Constraints', 'table_column_type pattern',
     'users_email_unique, orders_user_id_fk',
     'constraint1, fk_12345, my_check',
     'Error messages reference name; self-documenting is critical'),
    ('Indexes',    'table_columns_idx / table_columns_ukey',
     'orders_user_id_idx, users_email_ukey',
     'idx1, index_123, my_index',
     '\\di+ shows names; meaningful names avoid DROP INDEX mistakes'),
]

print('=== PostgreSQL Naming Conventions ===')
print()
print(f'{"Aspect":<14} {"Rule":<35} {"Good":<35} {"Avoid"}')
print('-' * 110)
for aspect, rule, good, bad, why in naming_rules:
    print(f'{aspect:<14} {rule:<35} {good:<35} {bad}')
    print(f'{"":14} Why: {why}')
    print()

## The "What If?" Experimentation Section

1. **What if you dump all tables in `public`?** Try `GRANT SELECT ON ALL TABLES IN SCHEMA public TO reporting_user` on a large app. It grants access to every table, including ones you did not intend. With a named `app` schema, you grant only `GRANT USAGE ON SCHEMA app TO reporting_user; GRANT SELECT ON ALL TABLES IN SCHEMA app TO reporting_user;`.

2. **What if you use quoted `"CamelCase"` identifiers?** Create `CREATE TABLE "UserAccounts" (...);` then try `SELECT * FROM useraccounts;` — it fails. You must always write `SELECT * FROM "UserAccounts"`. This cascades into every ORM query, migration tool, and monitoring dashboard forever.

3. **What if you store large images as BYTEA without setting STORAGE EXTERNAL?** PostgreSQL will try to compress JPEG data (which is already compressed) before storing externally. Set `ALTER TABLE documents ALTER COLUMN image_data SET STORAGE EXTERNAL;` to skip the wasteful compression attempt.

In [ ]:
# Experiment: search_path resolution simulation

search_path = ['app', 'extensions', 'public']

schema_objects = {
    'app':        ['users', 'orders', 'products', 'order_item'],
    'extensions': ['gen_random_uuid', 'crypt', 'digest'],
    'public':     ['users', 'legacy_data'],  # Note: 'users' also in public
}

def resolve_name(name, path, objects):
    for schema in path:
        if name in objects.get(schema, []):
            return schema
    return None

print(f'=== search_path: {" -> ".join(search_path)} ===')
print()

test_names = ['users', 'orders', 'legacy_data', 'nonexistent', 'gen_random_uuid']

for name in test_names:
    found_in = resolve_name(name, search_path, schema_objects)
    if found_in:
        print(f'  {name:<22} -> {found_in}.{name}')
        # Check if shadowed
        shadowed = [
            s for s in search_path[search_path.index(found_in)+1:]
            if name in schema_objects.get(s, [])
        ]
        if shadowed:
            print(f'  {"":22}    WARNING: shadows {shadowed[0]}.{name}')
    else:
        print(f'  {name:<22} -> ERROR: relation "{name}" does not exist')

print()
print('Lesson: Use schema-qualified names in production: app.users, not users')

## Real-World Project Link

**Mini project:** Audit an existing schema using system catalog queries.

Write queries that produce a schema health report:
1. All tables in `app` schema with sizes and estimated row counts
2. Foreign key columns NOT backed by an index on the child table
3. Tables missing `created_at` or `updated_at` columns
4. Constraints with auto-generated names (containing numeric suffixes)
5. Unused indexes (idx_scan = 0) that are candidates for removal

This is the kind of audit a new team member runs on day one, and the report a DBA runs before every major release.

## Chapter Summary & Cheat Sheet

### What you learned
- Cluster > Database > Schema > Object — cross-database queries need postgres_fdw
- Revoke `public` schema default privileges; create an `app` schema
- `pg_catalog` for operational scripts; `information_schema` for portable tooling
- TOAST transparently handles large values; set EXTERNAL for pre-compressed data
- Consistent `snake_case` with named constraints is the foundation of maintainable schemas

### Cheat sheet
```sql
-- Schema security baseline
REVOKE CREATE ON SCHEMA public FROM PUBLIC;
CREATE SCHEMA IF NOT EXISTS app;
ALTER DATABASE appdb SET search_path TO app, public;

-- Table sizes
SELECT relname, pg_size_pretty(pg_total_relation_size(oid))
FROM pg_class WHERE relkind = 'r' ORDER BY 2 DESC LIMIT 10;

-- Unused indexes
SELECT indexrelid::regclass, idx_scan
FROM pg_stat_user_indexes WHERE idx_scan = 0;

-- TOAST strategy
SELECT attname, CASE attstorage
  WHEN 'x' THEN 'EXTENDED' WHEN 'e' THEN 'EXTERNAL'
  WHEN 'p' THEN 'PLAIN'    WHEN 'm' THEN 'MAIN' END
FROM pg_attribute
WHERE attrelid = 'app.documents'::regclass AND attnum > 0;
```

## Practice Prompts

1. Explain why you cannot JOIN tables from two different PostgreSQL databases. What extension would you use if needed, and what are the performance implications?
2. What is `search_path`? Give an example of a subtle bug that `search_path` misconfiguration can cause.
3. Describe the four TOAST storage strategies. When would you explicitly set a column to EXTERNAL storage?
4. Write a `pg_catalog` query that lists all foreign key columns in the `app` schema and indicates whether each has a supporting index on the child table.
5. Why are quoted (`"CamelCase"`) identifiers considered an anti-pattern? What would you do if you inherited a schema that used them?

<div style='width:100%; display:flex; justify-content:space-between; align-items:center; margin: 1em 0;'>
  <a href='3.%20first_steps_with_psql.ipynb' style='padding:6px 14px; background:#007bff; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>← Previous</a>
  <a href='../TOC.md' style='padding:6px 14px; background:#6c757d; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>📚 Table of Contents</a>
  <a href='../2.%20SQL_essentials/5.%20data_types_done_right.ipynb' style='padding:6px 14px; background:#28a745; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>Next →</a>
</div>